In [1]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

!pip install chromadb rank_bm25 sentence-transformers underthesea -q

import json
import pickle
import chromadb
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from underthesea import word_tokenize
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 61.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 60.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 92.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 73.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [2]:
# ── Config ────────────────────────────────────────────────────────────────────
CONTEXTS_PATH     = "/kaggle/input/datasets/khanhtamnd/viet-rag-contexts/contexts.jsonl"
CHROMA_PERSIST    = "/kaggle/working/chroma_db"
CHROMA_COLLECTION = "virag_chunks"
BM25_PATH         = "/kaggle/working/bm25_index.pkl"
EMBEDDING_MODEL   = "Alibaba-NLP/gte-multilingual-base"
EMBED_BATCH_SIZE  = 64
UPSERT_BATCH      = 5000
MAX_LENGTH        = 512
DEVICE            = "cuda"

In [3]:
# ── 1. Load chunks ────────────────────────────────────────────────────────────
print("Loading chunks...")
chunks = []
with open(CONTEXTS_PATH, encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))
print(f"Loaded {len(chunks):,} chunks")

texts     = [c["text"]     for c in chunks]
ids       = [c["chunk_id"] for c in chunks]
metadatas = [{"title": c["title"], "source_split": c["source_split"]} for c in chunks]

Loading chunks...
Loaded 17,040 chunks


In [4]:
# ── 2. Load model directly via transformers (bypasses sentence-transformers
#       wrapper that passes unpad_inputs=True causing the ROPE overflow) ───────
print(f"\nLoading tokenizer and model: {EMBEDDING_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL, trust_remote_code=True)
model = AutoModel.from_pretrained(
    EMBEDDING_MODEL,
    trust_remote_code=True,
    ignore_mismatched_sizes=True,
).to(DEVICE)
model.eval()
print("Model loaded ✓")


Loading tokenizer and model: Alibaba-NLP/gte-multilingual-base


config.json: 0.00B [00:00, ?B/s]

configuration.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Alibaba-NLP/new-impl:
- configuration.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

A new version of the following files was downloaded from https://huggingface.co/Alibaba-NLP/new-impl:
- modeling.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/611M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

NewModel LOAD REPORT from: Alibaba-NLP/gte-multilingual-base
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded ✓


In [5]:
# ── 3. Embed with manual batching ─────────────────────────────────────────────
def mean_pool(token_embeddings, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)

def encode_texts(texts, batch_size=EMBED_BATCH_SIZE):
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        ).to(DEVICE)
        
        # Bypass the corrupted position_ids buffer
        seq_len = encoded["input_ids"].shape[1]
        encoded["position_ids"] = torch.arange(seq_len, dtype=torch.long, device=DEVICE).unsqueeze(0).expand(len(batch), -1)
        
        with torch.no_grad():
            out = model(**encoded, unpad_inputs=False)  
            
        # --- FIX: Sanitize NaNs generated by masked padding tokens ---
        hidden_states = torch.nan_to_num(out.last_hidden_state, nan=0.0)
        # -------------------------------------------------------------
        
        emb = mean_pool(hidden_states, encoded["attention_mask"])
        emb = torch.nn.functional.normalize(emb, p=2, dim=1)
        all_embeddings.append(emb.cpu().float().numpy())
        
    return np.vstack(all_embeddings)

print(f"\nEmbedding {len(texts):,} chunks...")
embeddings = encode_texts(texts)
print(f"Embeddings shape: {embeddings.shape} ✓")


Embedding 17,040 chunks...


Embedding: 100%|██████████| 267/267 [00:31<00:00,  8.55it/s]

Embeddings shape: (17040, 768) ✓


In [6]:
# ── 4. Build ChromaDB index ───────────────────────────────────────────────────
print(f"\nBuilding ChromaDB index at {CHROMA_PERSIST}...")
client = chromadb.PersistentClient(path=CHROMA_PERSIST)
try:
    client.delete_collection(CHROMA_COLLECTION)
    print("  (Deleted existing collection)")
except Exception:
    pass

collection = client.create_collection(
    name=CHROMA_COLLECTION,
    metadata={"hnsw:space": "cosine"},
)

print(f"Upserting {len(chunks):,} vectors...")
for i in tqdm(range(0, len(chunks), UPSERT_BATCH), desc="Upserting"):
    collection.upsert(
        ids=ids[i:i+UPSERT_BATCH],
        embeddings=embeddings[i:i+UPSERT_BATCH].tolist(),
        documents=texts[i:i+UPSERT_BATCH],
        metadatas=metadatas[i:i+UPSERT_BATCH],
    )
print(f"ChromaDB done ✓  ({collection.count():,} vectors stored)")


Building ChromaDB index at /kaggle/working/chroma_db...
Upserting 17,040 vectors...


Upserting: 100%|██████████| 4/4 [00:21<00:00,  5.35s/it]

ChromaDB done ✓  (17,040 vectors stored)


In [7]:
# ── 5. Build BM25 index ───────────────────────────────────────────────────────
print(f"\nBuilding BM25 index over {len(chunks):,} chunks...")
tokenised = [
    word_tokenize(t.lower(), format="text").split()
    for t in tqdm(texts, desc="Tokenising")
]
bm25 = BM25Okapi(tokenised)
payload = {
    "bm25":      bm25,
    "chunk_ids": ids,
    "texts":     texts,
    "metadatas": [{"title": c["title"]} for c in chunks],
}
with open(BM25_PATH, "wb") as f:
    pickle.dump(payload, f)
print(f"BM25 done ✓  saved to {BM25_PATH}")


Building BM25 index over 17,040 chunks...


Tokenising: 100%|██████████| 17040/17040 [00:58<00:00, 291.26it/s]


BM25 done ✓  saved to /kaggle/working/bm25_index.pkl


In [8]:
# ── 6. Verify ─────────────────────────────────────────────────────────────────
print("\n── Verification ──────────────────────────────────────────")
print(f"ChromaDB vectors : {collection.count():,}")
print(f"BM25 unique terms: {len(bm25.idf):,}")
print(f"Embeddings shape : {embeddings.shape}")
chroma_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, fns in os.walk(CHROMA_PERSIST) for f in fns
)
print(f"chroma_db/ size  : {chroma_size/1e6:.1f} MB")
print(f"bm25_index size  : {os.path.getsize(BM25_PATH)/1e6:.1f} MB")
print("\n✓ All done. Download chroma_db/ and bm25_index.pkl from /kaggle/working/")


── Verification ──────────────────────────────────────────
ChromaDB vectors : 17,040
BM25 unique terms: 39,437
Embeddings shape : (17040, 768)
chroma_db/ size  : 119.7 MB
bm25_index size  : 17.8 MB

✓ All done. Download chroma_db/ and bm25_index.pkl from /kaggle/working/


In [9]:
del client  # triggers __del__ -> flushes SQLite WAL
import shutil
shutil.make_archive('/kaggle/working/chroma_db', 'zip', '/kaggle/working/chroma_db')

'/kaggle/working/chroma_db.zip'